# Tính năng 1 – Trang **Dữ liệu huấn luyện**

Trang này cho biết mô hình đã học từ những tin tuyển dụng nào: bắt đầu từ bao nhiêu tin, bỏ đi bao nhiêu và vì sao, rồi chia dữ liệu ra sao.

Mỗi phần đi theo cùng một mạch để dễ theo dõi: **gặp vấn đề gì → để nguyên thì hỏng ở đâu → nhóm chọn cách nào → vì sao → phải đánh đổi gì**.

1. Dữ liệu gốc có những lỗi gì (đếm lại trên file gốc).
2. Phát hiện quan trọng nhất: **2.400 tin do máy sinh ra**, không phải tin thật.
3. Cách đọc các kiểu ghi lương khác nhau, chạy trên ví dụ thật.
4. Lọc dữ liệu và chia dữ liệu **theo công ty**.

> **Từ cần biết:** *tập học (train)* là phần dữ liệu mô hình được xem để học; *tập chọn (valid)* dùng để so và chọn mô hình; *tập kiểm tra (test)* chỉ mở một lần ở cuối để chấm điểm thật. *Trung vị* là giá trị đứng giữa khi xếp từ thấp đến cao.

Chi tiết về cách gán từng nhãn nằm ở notebook `gan_nhan_du_lieu.ipynb`. Mã nguồn: `src/career_market/data/rules.py`, `src/career_market/data/dataset.py`, `pages/dataset.py`.

In [ ]:
# Thiết lập: dùng đúng thư viện của ứng dụng (job_market/src), dữ liệu và mô hình đã lưu.
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "career_market").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from career_market.ui import charts  # thư viện biểu đồ chọn backend "Agg" (vẽ cho web)…
%matplotlib inline
# …nên bật lại chế độ inline để biểu đồ hiện trong notebook.
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Ba nguồn dữ liệu gốc và lỗi của chúng

In [ ]:
import re
import unicodedata
from career_market.paths import data_root

RAW = data_root() / "raw"
topcv = pd.read_csv(RAW / "topcv_jobs.csv")
posts = pd.read_csv(RAW / "vietnam_it_job_posts_v1.csv")
cb = pd.read_csv(RAW / "vietnamese-job-posting.csv")
display(pd.DataFrame({"Bộ dữ liệu": ["TopCV 2026", "IT Job Posts 2026", "CareerBuilder 2023"],
                      "Số dòng": [len(topcv), len(posts), len(cb)], "Số cột": [topcv.shape[1], posts.shape[1], cb.shape[1]]}))

raw_salary = pd.concat([topcv["salary"], posts["salary"], cb["salary"]]).fillna("").astype(str)
# Dữ liệu thô viết dấu tiếng Việt theo hai kiểu Unicode (dựng sẵn và tổ hợp): chuẩn hóa NFC trước khi so khớp.
salary = raw_salary.map(lambda v: unicodedata.normalize("NFC", v)).str.lower()
problems = {
    "Ghi 'thỏa thuận / thương lượng / cạnh tranh'": salary.str.contains("thỏa thuận|thoả thuận|thương lượng|cạnh tranh"),
    "Dấu tiếng Việt kiểu tổ hợp (nhìn giống, khác byte)": raw_salary != raw_salary.map(lambda v: unicodedata.normalize("NFC", v)),
    "Lỗi chính tả 'triẹu'": salary.str.contains("triẹu"),
    "Lương bằng USD": salary.str.contains(r"\$|usd"),
    "Lương theo năm": salary.str.contains(r"/\s*năm"),
    "Khoảng giả '1 triệu - 40 triệu'": salary.str.contains(r"^1 tri[ẹệ]u - 40 tri[ẹệ]u"),
}
pd.DataFrame({"Vấn đề": problems.keys(), "Số dòng": [int(m.sum()) for m in problems.values()]}).assign(
    **{"Tỉ lệ": lambda t: (t["Số dòng"] / len(salary) * 100).round(1).astype(str) + "%"})

**Khoảng lương giả "1 – 40 triệu".** Đây là giá trị điền sẵn của biểu mẫu đăng tin, không phải mức lương thật. Điểm giữa là 20,5 triệu, trông rất bình thường, nên máy không báo lỗi mà âm thầm học sai. Đây là kiểu lỗi nguy hiểm nhất. Cách xử lý: bỏ mọi khoảng lương mà mức cao gấp hơn 3 lần mức thấp.

## 2. Phát hiện lớn nhất: 2.400 tin do máy sinh ra

**Dấu hiệu đầu tiên:** ở ba trang TopCV, VietnamWorks, CareerViet, ngày đăng tin trải đều từng năm từ 2018 đến 2025. Thị trường thật không bao giờ đều như vậy. Lúc đầu nhóm chỉ nghĩ cột ngày bị hỏng; xem kỹ hơn thì thấy cả nhóm tin này có chung các dấu hiệu dưới đây.

In [ ]:
from career_market.data.rules import is_synthetic

synthetic = posts["source_url"].map(is_synthetic)
print("Số tin có đường dẫn dạng …-hist…:", int(synthetic.sum()))
display(posts[synthetic]["source_platform"].value_counts().rename("Số tin theo nền tảng").to_frame())
display(pd.DataFrame({
    "Tin sinh tự động": [posts[synthetic]["requirements"].notna().mean() * 100],
    "Tin thật": [posts[~synthetic]["requirements"].notna().mean() * 100],
}, index=["% có mô tả yêu cầu"]))
posts[synthetic][["title", "source_url", "skills"]].head(5)

In [ ]:
years = pd.DataFrame({
    "Tin sinh tự động": posts[synthetic]["posted_at"].astype(str).str[:4].value_counts(),
    "Tin thật": posts[~synthetic]["posted_at"].astype(str).str[:4].value_counts(),
}).fillna(0).sort_index()
ax = years.plot.bar(figsize=(9, 3.5), color=["#f97316", "#2563eb"], title="Số tin theo năm đăng ghi trong dữ liệu")
ax.set(xlabel="", ylabel="Số tin");

- **Vấn đề:** 2.400 tin (đúng 800 tin mỗi trang) có đường dẫn chứa `-hist`, ngày đăng trải đều, danh sách kỹ năng ngẫu nhiên (ví dụ tin Android lại đòi Swift và React) và không có mô tả công việc. Trước khi loại, chúng chiếm 54% số tin có lương.
- **Để nguyên thì sao:** tin giả dễ đoán hơn tin thật, nên điểm của mô hình đẹp một cách giả tạo. Mô hình cũ sai trung bình 16,5% khi chấm trên tập có lẫn tin giả, nhưng 18,0% khi chỉ chấm tin thật. Giống như ôn thi bằng bộ đề do máy bịa ra.
- **Nhóm chọn:** loại hẳn những tin này (thay vì chỉ sửa cột ngày hay đánh dấu).
- **Đánh đổi:** mất hơn nửa dữ liệu; tập kiểm tra cuối chỉ còn 288 tin.

## 3. Đọc lương – mỗi quy tắc đều có lý do

| Cách ghi trong tin | Cách xử lý | Vì sao |
|---|---|---|
| "15 - 25 triệu" | lấy điểm giữa: 20 triệu | Nhà tuyển dụng thường nói "lương khoảng 20 triệu"; mức cao là mức trần |
| "Tới 30 triệu" / "Từ 20 triệu" | đoán đầu còn lại theo **thói quen ghi lương của từng trang** (học từ tin ghi đủ hai đầu) | Mỗi trang ghi khoảng rộng hẹp khác nhau; một hệ số cố định không đúng cho mọi trang |
| Đô la / lương theo năm | đổi theo tỷ giá của năm đó / chia 12 | Tránh "1.000 USD" bị hiểu thành 1.000 triệu |
| Mức cao gấp hơn 3 lần mức thấp | bỏ | Một vị trí hiếm khi chênh nhiều thế; khoảng quá rộng là cách né công bố lương |

In [ ]:
import json
from career_market.data.rules import parse_salary
from career_market.paths import dataset_path

summary = json.loads(dataset_path().with_suffix(".json").read_text(encoding="utf-8"))
ratios = summary["one_sided_ratio"]
display(pd.Series(ratios, name="Hệ số cao/thấp (học trên train)").to_frame())

examples = ["15 - 25 triệu", "Lương: 10 Tr - 15 Tr VND", "$ 1,000-1,500 /tháng", "240tr-360tr ₫/năm", "1 triẹu - 40 triẹu/tháng", "Thỏa thuận"]
rows = [{"Chuỗi gốc": t, "Kết quả (thấp, cao, trung điểm)": parse_salary(t, 2026) or "loại khỏi nhãn"} for t in examples]
for text in ["Tới 30 triệu", "Từ 20 triệu"]:
    for source in ["IT Jobs 2026 (itviec)", "TopCV 2026"]:
        rows.append({"Chuỗi gốc": f"{text}  [{source}]", "Kết quả (thấp, cao, trung điểm)": parse_salary(text, 2026, ratio=ratios[source])})
pd.DataFrame(rows)

## 4. Lọc dữ liệu và chia theo công ty

**Vì sao chia theo công ty?** Một công ty thường đăng hàng chục tin gần giống nhau. Nếu chia ngẫu nhiên từng tin, mô hình chỉ cần "nhớ" công ty là đoán đúng, giống như được ôn đúng đề sẽ thi. Nhưng người dùng thật thường hỏi về **công ty mới**, nên tập kiểm tra phải gồm những công ty mô hình chưa từng thấy. Mỗi công ty được xếp vào một tập cố định theo tên (dùng hàm băm), nên chạy lại bao nhiêu lần cũng ra cùng một cách chia.

In [ ]:
from career_market.data.dataset import load_dataset

funnel = summary["funnel"]
steps = pd.DataFrame({
    "Bước": ["Đọc ba nguồn (CareerBuilder chỉ nhóm CNTT)", "Sau khi loại tin sinh tự động", "Có lương đọc được", "Thuộc 23 vai trò CNTT", "Sau khử trùng lặp"],
    "Số tin": [funnel["raw"], funnel["raw"] - funnel["synthetic_removed"], funnel["has_salary"], funnel["it_role"], funnel["deduplicated"]],
})
steps["Tỉ lệ"] = (steps["Số tin"] / funnel["raw"] * 100).round(1).astype(str) + "%"
display(steps)

frame = load_dataset()
table = frame.groupby("split").agg(**{"Số tin": ("row_id", "size"), "Số công ty": ("company_key", "nunique")}).loc[["train", "valid", "test"]]
table["Tỉ lệ tin"] = (table["Số tin"] / len(frame) * 100).round(1).astype(str) + "%"
display(table)
groups = {s: set(frame.loc[frame["split"] == s, "company_key"]) for s in ["train", "valid", "test"]}
print("Công ty xuất hiện ở hai tập:", len(groups["train"] & groups["valid"]) + len(groups["train"] & groups["test"]) + len(groups["valid"] & groups["test"]))
print("Còn tin sinh tự động trong bộ dữ liệu sạch:", int(frame["source_url"].map(is_synthetic).sum()))

Hai biểu đồ dưới đây giống hệt hai biểu đồ trên trang Dữ liệu huấn luyện.

In [ ]:
charts.split_by_role(frame);

In [ ]:
charts.salary_distribution(frame);

## Kết luận
- Mỗi lỗi của dữ liệu gốc có một cách xử lý rõ ràng và được kiểm thử tự động.
- Bài học lớn nhất: *nhiều dữ liệu chỉ có ích khi dữ liệu là thật*. Bỏ 2.400 tin giả làm điểm của mô hình kém đi, nhưng là điểm trung thực.
- Chia theo công ty là quyết định quan trọng nhất về cách đánh giá: mọi con số đều đo trên những công ty mô hình chưa từng gặp.